In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [18]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"

log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
# session.close()

# engine.dispose()

### The Network Name in db and Ted's spread

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-ORG NAME CHANGES-Edits.xlsx', header = 0)   # pandas automatically uses openpyxl

df["network_id"] = pd.to_numeric(df["network_id"], errors="coerce").astype("Int64")

df_rename = df[df["status"].str.strip().isin(["Rename"])]
df_new = df[df["status"].str.strip().isin(["new"])]



In [5]:
df_rename

,network_id,network_name,description,Current,Suggested Network name,status
0,10,AGRI,BC Ministry of Agriculture and Foods,AGRI,BC AGRIWeather,Rename
3,5,BCH,BC Hydro - Hydrometric Monitoring Program,BCH,BCH-GSO-HMP,Rename
5,36,CRD,Capital Regional District Water Supply Area,CRD,CRD-WS,Rename
10,19,EC_raw,Environment and Climate Change Canada (raw obs...,EC_raw,EC_Raw,Rename
11,9,ENV-AQN,BC Ministry of Environment and Parks - Air Qua...,ENV-AQN,BC-Air,Rename
12,14,ENV-ASP,BC Ministry of Environment and Parks - Automat...,ENV-ASP,BC-Snow,Rename
13,11,FLNRO-FERN,BC Ministry of Forests - Forest Ecosystems Res...,FLNRO-FERN,BC-FERN,Rename
15,12,FLNRO-WMB,BC Ministry of Forests - Wildfire Service Network,FLNRO-WMB,BC-FWx,Rename
19,15,MVan,MetroVancouver - Water Supply,MVan,MVRD-WS,Rename
22,2,MoTIe,Ministry of Transportation and Transit - Autom...,MoTIe,BC-TRAN,Rename


In [6]:
df_new

,network_id,network_name,description,Current,Suggested Network name,status
4,<NA>,NaN,BC Hydro - Site C,BCH,BCH-SiteC,new
14,<NA>,NaN,BC Timber Sales,FLNRO-WMB,BC-TS,new
18,137,MVRD,Metro Vancouver - Air Quality,MVRD,MVRD-AQ,new
24,<NA>,NaN,Nav Canada,NaN,NAVCAN,new
25,<NA>,NaN,Parks Canada - Avalanche,NaN,PC-Aval,new
26,<NA>,NaN,Parks Canada - Wildfire Service,NaN,PC-FWx,new
28,<NA>,NaN,Prince Rupert Port Authority,NaN,PRPA,new


### Network rename

In [20]:
update_station_sql = sa.text("""
UPDATE meta_network
SET network_display_name = :new_network_name
WHERE network_id = :old_network_id
""")

with engine.begin() as conn:
    for _, row in df_rename.iterrows():
        result = conn.execute(
            update_station_sql,
            {
                "old_network_id": row["network_id"],
                "new_network_name": row["Suggested Network name"],
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {row['network_id']}: "
                f"{row['network_name']} → {row['Suggested Network name']}"
            )



✅ Updated station 10: AGRI → BC AGRIWeather
✅ Updated station 5: BCH → BCH-GSO-HMP
✅ Updated station 36: CRD → CRD-WS
✅ Updated station 19: EC_raw → EC_Raw
✅ Updated station 9: ENV-AQN → BC-Air
✅ Updated station 14: ENV-ASP → BC-Snow
✅ Updated station 11: FLNRO-FERN → BC-FERN
✅ Updated station 12: FLNRO-WMB → BC-FWx
✅ Updated station 15: MVan → MVRD-WS
✅ Updated station 2: MoTIe → BC-TRAN
✅ Updated station 3: MoTIm → BC-TRANm


In [21]:
update_description_sql = sa.text("""
UPDATE meta_network
SET description = :description
WHERE network_display_name = :new_network_name
""")


for _, row in df_rename.iterrows():
    session.execute(
        update_description_sql,
        {
            "description": row["description"],
            "new_network_name": row["Suggested Network name"]
        }
    )

session.commit()

#### Insert new network

In [22]:
df_new

,network_id,network_name,description,Current,Suggested Network name,status
4,<NA>,NaN,BC Hydro - Site C,BCH,BCH-SiteC,new
14,<NA>,NaN,BC Timber Sales,FLNRO-WMB,BC-TS,new
18,137,MVRD,Metro Vancouver - Air Quality,MVRD,MVRD-AQ,new
24,<NA>,NaN,Nav Canada,NaN,NAVCAN,new
25,<NA>,NaN,Parks Canada - Avalanche,NaN,PC-Aval,new
26,<NA>,NaN,Parks Canada - Wildfire Service,NaN,PC-FWx,new
28,<NA>,NaN,Prince Rupert Port Authority,NaN,PRPA,new


In [23]:
from pycds import Network
help(Network)


Help on class Network in module pycds.orm.tables:

class Network(sqlalchemy.orm.decl_api.Base)
 |  Network(**kwargs)
 |
 |  This class maps to the table which represents various `networks` of
 |  data for the Climate Related Monitoring Program. There is one
 |  network row for each data provider, typically a BC Ministry, crown
 |  corporation or private company.
 |
 |  Method resolution order:
 |      Network
 |      sqlalchemy.orm.decl_api.Base
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, **kwargs) from sqlalchemy.orm.instrumentation
 |      A simple constructor that allows initialization from kwargs.
 |
 |      Sets attributes on the constructed instance using the names and
 |      values in ``kwargs``.
 |
 |      Only keys that are present as
 |      attributes of the instance's class are allowed. These could be,
 |      for example, any mapped columns or relationships.
 |
 |  __str__(self)
 |      Return str(self).
 |
 |  ------------------------------

In [14]:
from pycds import Network

# List all columns
print(Network.__table__.columns.keys())

['network_id', 'network_name', 'description', 'virtual', 'publish', 'col_hex', 'contact_id', 'mod_time', 'mod_user']


In [24]:
existing = {
    n.name for n in session.query(Network.name).all()
}

# existing
for _, row in df_new.iterrows():
    name = row["Suggested Network name"]

    if name in existing:
        continue

    net = Network(
        name=name,
        long_name=row["description"],
        publish=True,
    )

    session.add(net)
    session.flush()
    existing.add(name)


session.commit()

In [14]:
# row = df_new.loc[
#     df_new["Suggested Network name"] == "MVRD-AQ"
# ].iloc[0]

# name = row["Suggested Network name"]

# existing = {
#     n.name for n in session.query(Network.name).all()
# }

# if name not in existing:
#     net = Network(
#         name=name,
#         long_name=row["description"],
#         publish=True,
#     )
#     session.add(net)
#     session.commit()
